# Case Study 1 — Gradient-Boosted Direction Classifier on US Equities

**Hypothesis:** A gradient-boosted classifier trained on technical, macro, and cross-sectional features can predict the *sign* of next-week US equity returns better than a coin flip — and the resulting long/short portfolio beats buy-and-hold after costs.

**The catch:** This is the recipe everyone tries first. We're going to evaluate it the way most blog posts don't:

- **Purged + embargoed walk-forward CV** so the label window can't leak into training.
- **Realistic transaction costs** at the bps level (commission + half-spread).
- **Deflated Sharpe Ratio** to account for the fact that we're one of many models being tested.
- **Bootstrap 95% CI** on annualised return — point estimates are vibes.
- **Per-regime breakdown** so a strategy that only worked in 2017 gets exposed.

**Reproducibility:** This notebook caches all yfinance/FRED downloads under `data_cache/` (date-keyed). On a fresh clone the first run pulls data; subsequent runs the same day are offline.

Required env vars: `FRED_API_KEY` (free at https://fred.stlouisfed.org/). Yahoo Finance needs no key.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from backtester.data.yfinance import fetch_daily as yf_fetch
from backtester.data.fred import fetch_series as fred_fetch
from backtester.data.universe import eligible_tickers, load_universe
from backtester.features import (
    log_returns, rolling_volatility, rsi, macd,
    macro_features, momentum_rank, vol_rank,
)
from backtester.eval.walkforward import walk_forward_splits
from backtester.eval.statistics import (
    annualised_sharpe, deflated_sharpe_ratio, bootstrap_ci,
)
from backtester.eval.costs import EQUITIES_LIQUID, apply_costs
from backtester.eval.regimes import trend_regimes, per_regime_metrics
from backtester.models import GBMClassifier

pd.options.display.float_format = '{:,.4f}'.format
plt.rcParams.update({'figure.figsize': (10, 4), 'axes.grid': True})
RNG = np.random.default_rng(42)

START = '2010-01-01'
END = '2024-12-31'
LABEL_HORIZON = 5  # next-5-day return
RETURN_COL = 'fwd_ret_5d'
TOP_BOTTOM_QUANTILE = 0.2  # long top 20%, short bottom 20%
N_SPLITS = 6
EMBARGO = 5

## 1. Load price data for the eligible universe

We use the bundled `samples/universe_us_liquid.csv` snapshot. Tickers are eligible from their `first_eligible` date onwards — this rules out (most) survivorship bias from the cross-sectional ranks.

In [ ]:
universe = load_universe()
tickers = sorted(universe['ticker'])
print(f'Universe size: {len(tickers)}')

prices = {}
for tkr in tickers:
    try:
        df = yf_fetch(tkr, start=START, end=END)
        prices[tkr] = df['adj_close'] if 'adj_close' in df.columns else df['close']
    except Exception as exc:
        print(f'  skip {tkr}: {exc}')

px = pd.DataFrame(prices).sort_index()
px = px.loc[~px.index.duplicated(keep='first')]
print(f'Price panel: {px.shape[0]} rows x {px.shape[1]} tickers, {px.index.min().date()} -> {px.index.max().date()}')

## 2. Build the feature panel

All features are leakage-free (enforced by `tests/test_features_leakage.py`). For each (date, ticker) we stack:

- **Single-asset technical:** 5/20/60-day momentum, 20-day vol, RSI, MACD-line.
- **Cross-sectional:** momentum rank (12-1), vol rank.
- **Macro:** VIX level + 5d change, T10Y2Y slope + change, BAA10Y credit spread + change. Each lagged 1 trading day.

In [ ]:
rets = px.pct_change()
log_rets = np.log(px).diff()

tech_blocks = []
for tkr in px.columns:
    s = px[tkr]
    block = pd.DataFrame({
        'mom_5':  s.pct_change(5),
        'mom_20': s.pct_change(20),
        'mom_60': s.pct_change(60),
        'vol_20': rolling_volatility(log_rets[tkr], window=20),
        'rsi_14': rsi(s, period=14),
        'macd_line': macd(s)['macd_line'],
    })
    block['ticker'] = tkr
    tech_blocks.append(block.reset_index().rename(columns={'index': 'datetime', 'Date': 'datetime'}))
tech_long = pd.concat(tech_blocks, ignore_index=True)
if 'datetime' not in tech_long.columns:
    tech_long = tech_long.rename(columns={tech_long.columns[0]: 'datetime'})
tech_long = tech_long.set_index(['datetime', 'ticker']).sort_index()

mom_panel = momentum_rank(px)
vol_panel = vol_rank(rets)

tech_long['mom_rank'] = mom_panel.stack().reindex(tech_long.index).values
tech_long['vol_rank'] = vol_panel.stack().reindex(tech_long.index).values

calendar = px.index
fred_series = {}
for sid in ['VIXCLS', 'T10Y2Y', 'BAA10Y']:
    try:
        fred_series[sid] = fred_fetch(sid)
    except RuntimeError as exc:
        print(f'FRED skipped ({sid}): {exc}')
macro = macro_features(fred_series, calendar) if fred_series else pd.DataFrame(index=calendar)

tech_with_macro = tech_long.join(macro, on='datetime') if not macro.empty else tech_long
tech_with_macro = tech_with_macro.dropna()
print(f'Feature rows after dropna: {len(tech_with_macro):,}')
tech_with_macro.head()

## 3. Build the label and align

Label = sign of next 5-day return. We hold predictions to a **per-day cross section** so each walk-forward fold sees full date-blocks (essential for the purge to be meaningful).

In [ ]:
fwd_ret = px.pct_change(LABEL_HORIZON).shift(-LABEL_HORIZON)
labels_long = fwd_ret.stack().rename(RETURN_COL)
labels_long.index.names = ['datetime', 'ticker']
y = (labels_long > 0).astype(int)

design = tech_with_macro.join(y.rename('y'), how='inner').join(labels_long, how='inner').dropna()
print(f'Design matrix: {len(design):,} rows')
feature_cols = [c for c in design.columns if c not in {'y', RETURN_COL}]
print('Features:', feature_cols)

## 4. Walk-forward training & out-of-sample predictions

We split the *date axis* (not the row axis) so each fold is a contiguous block of trading days. Purge horizon = label horizon (5). Embargo = 5 trading days.

In [ ]:
dates = design.index.get_level_values('datetime').unique().sort_values()
n_days = len(dates)
splits = list(walk_forward_splits(
    n=n_days, n_splits=N_SPLITS,
    label_horizon=LABEL_HORIZON, embargo=EMBARGO,
))

X = design[feature_cols]
y_full = design['y']

oos_proba = pd.Series(np.nan, index=design.index, dtype=float)
for i, (train_di, test_di) in enumerate(splits, 1):
    train_dates = dates[train_di]
    test_dates = dates[test_di]
    train_mask = design.index.get_level_values('datetime').isin(train_dates)
    test_mask = design.index.get_level_values('datetime').isin(test_dates)
    if train_mask.sum() < 5_000 or test_mask.sum() < 200:
        continue
    model = GBMClassifier()
    model.fit(X[train_mask], y_full[train_mask])
    oos_proba[test_mask] = model.predict_proba(X[test_mask])
    print(f'Fold {i}: train={train_mask.sum():,} test={test_mask.sum():,} '
          f'dates {train_dates.min().date()}->{train_dates.max().date()} | '
          f'{test_dates.min().date()}->{test_dates.max().date()}')

oos = design.loc[oos_proba.dropna().index].copy()
oos['proba'] = oos_proba.dropna()
print(f'OOS rows: {len(oos):,}')

## 5. Build the long/short portfolio

Each rebalance day, go **long** the top 20% of probabilities, **short** the bottom 20%, equal-weight, dollar-neutral. Costs applied on |Δ position|.

In [ ]:
def long_short_position(df: pd.DataFrame, q: float) -> pd.Series:
    upper = df['proba'].quantile(1 - q)
    lower = df['proba'].quantile(q)
    pos = pd.Series(0.0, index=df.index)
    pos[df['proba'] >= upper] = 1.0
    pos[df['proba'] <= lower] = -1.0
    n_long = (pos > 0).sum()
    n_short = (pos < 0).sum()
    if n_long:
        pos[pos > 0] = 1.0 / n_long
    if n_short:
        pos[pos < 0] = -1.0 / n_short
    return pos

positions = oos.groupby(level='datetime', group_keys=False).apply(
    lambda df: long_short_position(df, TOP_BOTTOM_QUANTILE)
)

# Strategy return per (date, ticker) = position * forward 5d return; daily-equivalent
# return is the cross-sectional sum (since positions are normalised).
per_row_ret = positions * oos[RETURN_COL] / LABEL_HORIZON
daily_gross = per_row_ret.groupby(level='datetime').sum().sort_index()

# Approximate turnover: book-level |delta-weight| summed over names per day.
wide_pos = positions.unstack('ticker').reindex(daily_gross.index).fillna(0.0)
book_turnover = wide_pos.diff().abs().sum(axis=1).fillna(wide_pos.abs().sum(axis=1).iloc[0])
daily_net = apply_costs(daily_gross, wide_pos.abs().sum(axis=1).clip(upper=2.0), EQUITIES_LIQUID)
# More accurate: charge book turnover directly
from backtester.eval.costs import BPS
daily_net = daily_gross - book_turnover * EQUITIES_LIQUID.per_turnover_bps * BPS

print(f'Daily strategy return series: {len(daily_net):,} rows')

## 6. Honest evaluation

Deflated Sharpe accounts for the fact that we're one of many candidates being tested — we plug in `n_trials=20` as a proxy for the implicit hyperparameter / feature combinations explored.

**Important:** if `dsr < 0.95`, we have **failed to reject** the null that this strategy is no better than the best of 20 random models. That's the conclusion most signals deserve.

In [ ]:
ret = daily_net.dropna().to_numpy()
sr = annualised_sharpe(ret)
dsr = deflated_sharpe_ratio(ret, n_trials=20)
ci = bootstrap_ci(ret, statistic=annualised_sharpe, n_resamples=1000, block_size=20, rng=RNG)

print(f'Annualised Sharpe (net of costs): {sr:.3f}')
print(f'Deflated Sharpe Ratio (n_trials=20): {dsr:.3f}')
print(f'Bootstrap 95% CI on Sharpe: [{ci.lower:.3f}, {ci.upper:.3f}]')
ann_ret = (1 + pd.Series(ret).mean()) ** 252 - 1
print(f'Approx annualised return: {ann_ret * 100:.2f}%')

## 7. Equity curve + drawdown vs. SPY benchmark

In [ ]:
spy = yf_fetch('SPY', start=START, end=END)
spy_close = spy['adj_close'] if 'adj_close' in spy.columns else spy['close']
spy_ret = spy_close.pct_change().reindex(daily_net.index).fillna(0.0)

equity_strat = (1 + daily_net).cumprod()
equity_spy = (1 + spy_ret).cumprod()

fig, ax = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
ax[0].plot(equity_strat, label='GBM long/short (net)', linewidth=1.4)
ax[0].plot(equity_spy, label='SPY buy & hold', linewidth=1.0, alpha=0.7)
ax[0].set_ylabel('Equity (×)')
ax[0].legend()
ax[0].set_title('Case 1 — GBM direction classifier on US equities')

running_max = equity_strat.cummax()
dd = equity_strat / running_max - 1
ax[1].fill_between(dd.index, dd.values, 0, color='crimson', alpha=0.4)
ax[1].set_ylabel('Drawdown')
plt.tight_layout()
plt.show()

## 8. Per-regime breakdown

Bull = SPY above its 200d SMA. A signal that only works in bulls (or in a single regime year) doesn't generalise.

In [ ]:
regimes = trend_regimes(spy_close, window=200).reindex(daily_net.index)
per_regime_sharpe = per_regime_metrics(
    daily_net, regimes,
    metric=lambda x: annualised_sharpe(x),
)
pd.Series(per_regime_sharpe, name='Sharpe by regime').to_frame()

## Discussion (fill in with observed numbers)

Re-run this notebook end-to-end and fill in:

- Did the **deflated Sharpe** stay above the meaningful-edge threshold (~0.95)?
- Was the bootstrap CI strictly above zero?
- Did the strategy make money in **both** bull and bear regimes?
- How does it compare to SPY net of costs?

If any answer is "no", that's the honest finding — and it's a more interesting result than a fake-positive backtest.